# Mimicry-pair recovery

Does visual similarity place published mimicry partners near each other? Dorsal against
dorsal: each tested species' dorsal UNICOM centroid queries the dorsal images of every
species in the collection, and each species scores its nearest dorsal image, the rule
`similarity run` uses for the site's list. Ventral images are neither queries nor
candidates.

Every species in the collection is ranked, so a partner outside the top ten still has a
measured rank. Only the pairs with dorsal images of both species in the collection are
tested; pairs come from `analyses/data/mimicry_pairs.csv`, with their publications and
DOIs. Run with the backend stopped.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
from analyses.helpers.mimicry import describe, mimicry_results, recovery_figure
from analyses.helpers.publication import export_figure, load_settings, publication_style

settings = load_settings(ROOT)
publication_style()

N_PERMUTATIONS = 10_000
SEED = 0

## Rank and test every pair

One dorsal centroid per query species, scored against every collection species by its
nearest dorsal image. The images beside each pair are the ones that set those scores:
each species' dorsal image nearest the other species' centroid.

In [ ]:
results = mimicry_results(settings, ROOT, N_PERMUTATIONS, SEED)
print(describe(results))

## Tested pairs and species

Tested species are counted by accepted name, so a published name the collection files under
another species counts once; every tested species is ranked against all species with dorsal
images.

In [ ]:
results.counts()

## Pairs and coverage

Species are MATCHED accepted names. A published name that is not itself accepted resolves
through the images recorded under it, when they agree on one accepted species (for example
*Adelpha bredowii* → *Limenitis bredowii*); `resolution` shows how each member resolved.
A pair is tested only when both members resolve to distinct species with dorsal images.
Pairs not in the collection are left out of the result tables; the summary above names
them and why.

In [ ]:
results.coverage

## Partner ranks

Each pair is queried in both directions. `recovery` marks pairs that are
mutual top ten: each species among the other's ten most similar.

In [ ]:
results.ranks.drop(columns=["mimicry_type"])

In [ ]:
results.recovery

## Permutation tests

The statistic is the mean partner percentile over both query directions of every tested
pair (1 = nearest, 0.5 = chance), equal to the mean of the pair means. Each permutation
keeps the queries and draws a new partner per row:

- **Random**: any other collection species.
- **Query congeners**: for pairs within one genus, another congener of the query, since
  close relatives look alike without mimicry; pairs across genera keep the random pool.
- **Partner congeners**: another congener of the partner, for every pair: does the query
  resemble its mimicry partner more than the partner's relatives? Rows whose partner genus
  has no other species are left out (`rows_used`).

Directions of one pair, and species shared by several pairs, are not independent; the
p-values describe these pairs against these nulls, not a population of mimicry pairs.

In [ ]:
results.permutations

## Pairs as the unit

One row per pair and null. A direction's p-value is the share of its null
candidates, with the partner itself, that sit at least as near the query as the partner:
an exact per-pair test that needs no pooling across pairs.

In [ ]:
results.pair_tests

## Leave one species out

Species shared by several pairs make those pairs dependent. Each test is repeated without
every pair holding one shared species.

In [ ]:
results.leave_one_out

## Combined figure

A) Partner rank by nearest dorsal image. Beside each pair are the two images that matched:
each species' dorsal image nearest the other species' centroid, first species farther from
the axis, with N, the species' dorsal image count, below it. Filled markers query with the
pair's first species, hollow ones with its second; the dashed line marks the site's top ten
and the dotted line the chance median. Shaded, bold pairs are mutual top ten: each species
is among the other's ten most similar. B) Observed mean partner percentile against each
null's mean and 95% interval. Pair labels end in the mimicry type: (M) Müllerian, (B)
Batesian, (M*) Müllerian or context-dependent.

In [ ]:
fig = recovery_figure(results, results.pictures(ROOT))
export_figure(fig, settings, "mimicry_recovery", results.exports())
plt.show()
plt.close(fig)